In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import sum
from pyspark.sql.functions import avg

In [0]:
print("Hello Databricks sasdasdasd")

In [0]:
# DBTITLE 1,Create a Spark DataFrame
data = [
    (101, "Rahul", "Laptop", 1, 50000),
    (102, "Priya", "Phone", 2, 30000),
    (103, "Arjun", "Laptop", 1, 60000),
    (104, "Rahul", "Mouse", 2, 1500),
    (105, "Sneha", "Tablet", 1, 20000)
]
columns = [
    "order_id",
    "customer",
    "product",
    "quantity",
    "price"
]

sales_df = spark.createDataFrame(data, columns)

display(sales_df)

In [0]:
# DBTITLE 1,Schema
sales_df.printSchema()
sales_df.count()

In [0]:
#select only the columns we need
sales_df.select(
    "order_id",
    "customer",
    "product"
).show()

In [0]:
#filter high value orders
high_value_orders = sales_df.filter(
    col("price") >= 20000
)

display(high_value_orders)

In [0]:
#calculate the actual order value:
sales_with_total = sales_df.withColumn(
    "total_amount",
    col("quantity") * col("price")
)

display(sales_with_total)

In [0]:
customer_sales = (
    sales_with_total
    .groupBy("customer")
    .sum("total_amount")
)

display(customer_sales)

In [0]:
#Validation
sales_df.printSchema()
sales_df.count()
sales_df.show()

In [0]:
#Inspect a DataFrame
sales_df.describe().show()

In [0]:
#Rename Column
sales_with_total = sales_with_total.withColumnRenamed(
    "customer",
    "customer_name"
)
sales_with_total.show()

In [0]:
#orderby
sales_with_total.orderBy(
    col("total_amount").desc()
).show()

In [0]:
df2 = (
    sales_df
      .filter(col("price") > 20000)
      .select("customer", "price")
)
df2.show()

In [0]:
df2.explain(True)

In [0]:

%sql
SHOW CATALOGS;
SHOW SCHEMAS IN harshadatabricksdebt;

In [0]:
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "harshadatabricksdebt.default.sales_bronze"
    )

In [0]:
bronze_df = spark.table(
    "harshadatabricksdebt.default.sales_bronze"
)

display(bronze_df)

In [0]:
%sql
SELECT *
FROM harshadatabricksdebt.default.sales_bronze;

In [0]:
%sql
DESCRIBE HISTORY harshadatabricksdebt.default.sales_bronze;

In [0]:
new_order = [
    (106, "Vikram", "Monitor", 1, 25000)
]

new_order_df = spark.createDataFrame(
    new_order,
    [
        "order_id",
        "customer",
        "product",
        "quantity",
        "price"
    ]
)

display(new_order_df)

In [0]:
new_order_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "harshadatabricksdebt.default.sales_bronze"
    )

In [0]:
%sql
SELECT *
FROM harshadatabricksdebt.default.sales_bronze
ORDER BY order_id;

In [0]:
from pyspark.sql.functions import col, count

bronze_df.groupBy("order_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1) \
    .show()

In [0]:
silver_df = bronze_df.dropDuplicates(
    ["order_id"]
)

silver_df = silver_df.withColumn(
    "total_amount",
    col("quantity") * col("price")
)
silver_df = silver_df.withColumnRenamed(
    "customer",
    "customer_name"
)
display(silver_df)
silver_df.printSchema()
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "harshadatabricksdebt.default.sales_silver"
    )

In [0]:

silver_check = spark.table(
    "harshadatabricksdebt.default.sales_silver"
)

silver_check.count()
silver_check.groupBy("order_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1) \
    .show()


In [0]:
display(
    silver_check.select(
        "order_id",
        "customer_name",
        "product",
        "quantity",
        "price",
        "total_amount"
    ).orderBy("order_id")
)

In [0]:
silver_check.selectExpr(
    "SUM(total_amount) AS total_revenue"
).show()

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(total_amount) AS total_revenue
FROM harshadatabricksdebt.default.sales_silver;

In [0]:
silver_df = spark.table(
    "harshadatabricksdebt.default.sales_silver"
)

display(silver_df)

In [0]:
gold_df = (
    silver_df
    .groupBy("customer_name")
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("total_sales")
    )
)

display(gold_df)

In [0]:

gold_df = (

    silver_df
    .groupBy("customer_name")
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("total_sales"),
        avg("total_amount").alias("average_order_value")
    )
)
gold_df=gold_df.orderBy(
    col("total_sales").desc()
)
display(gold_df)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "harshadatabricksdebt.default.customer_sales_gold"
    )

In [0]:
gold_check = spark.table(
    "harshadatabricksdebt.default.customer_sales_gold"
)

display(gold_check)

In [0]:
gold_check.selectExpr(
    "SUM(total_sales) AS total_revenue"
).show()

In [0]:
%sql

DESCRIBE HISTORY harshadatabricksdebt.default.sales_bronze;

SELECT COUNT(*) AS rows
FROM harshadatabricksdebt.default.sales_bronze
VERSION AS OF 0;

SELECT COUNT(*) AS rows
FROM harshadatabricksdebt.default.sales_bronze
VERSION AS OF 1;

SELECT COUNT(*) AS rows
FROM harshadatabricksdebt.default.sales_bronze;